In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent.parent))
from notebooks.eval.shared_setup import *

# Part III: Fairness & Popularity Bias Analysis

Now that we know *how well* each strategy retrieves, we investigate *how fairly* it retrieves.

- **Expected vs. Retrieved popularity**: do retrieved documents come from the same popularity tier as the question target?
- **Popularity preference**: when the system retrieves the wrong document, is that document *more popular* than the correct one?
- **Where do wrong documents come from?** — per-decile origin analysis

### 7. Expected vs. Retrieved Popularity Heatmap

Each cell $(i, j)$ shows the fraction of questions whose target is in popularity decile $i$ but whose **top-1 retrieved document** falls into corpus-popularity decile $j$.  
A perfectly unbiased system concentrates mass on the diagonal (green squares).

### 7a. Expected vs. Retrieved Heatmap by Company — BM25 minus Approximation

Difference heatmap per company: **BM25 − Approximation**. Positive values (red) mean BM25 retrieves relatively more from that cell; negative values (blue) mean the approximation strategy retrieves relatively more.

In [ ]:
from src.metrics.metrics import pick_group_col as _pick_group_col
from src.metrics.bias_utils import build_heatmap_pct as _build_heatmap_pct

from src.metrics.decile_utils import decile_col_for, boundaries_for



decile_col = decile_col_for(DECILE_MODE)
boundaries = boundaries_for(DECILE_MODE, boundaries_uw, boundaries_cw)

df_bm25 = results_by_strategy['bm25']
df_approx = results_by_strategy['approximation']
group_col = _pick_group_col(df_bm25)
if not group_col:
    print('⚠️ No grouping column found')
else:
    groups = sorted(set(df_bm25[group_col].unique()) | set(df_approx[group_col].unique()))
    n_groups = len(groups)
    n_cols = min(3, n_groups)
    n_rows = (n_groups + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5*n_cols, 4.5*n_rows))
    if n_rows * n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)
    abs_max = 0
    diff_heatmaps = {}
    for group in groups:
        sub_bm25 = df_bm25[df_bm25[group_col] == group] if group in df_bm25[group_col].values else None
        sub_approx = df_approx[df_approx[group_col] == group] if group in df_approx[group_col].values else None
        hm_bm25 = _build_heatmap_pct(sub_bm25, decile_col, boundaries) if sub_bm25 is not None and len(sub_bm25) > 0 else np.zeros((10, 10))
        hm_approx = _build_heatmap_pct(sub_approx, decile_col, boundaries) if sub_approx is not None and len(sub_approx) > 0 else np.zeros((10, 10))
        diff = hm_bm25 - hm_approx
        diff_heatmaps[group] = diff
        abs_max = max(abs_max, np.abs(diff).max())
    abs_max = abs_max or 1.0
    for idx, group in enumerate(groups):
        r, c = divmod(idx, n_cols)
        ax = axes[r, c]
        diff = diff_heatmaps[group]
        im = ax.imshow(diff, aspect='auto', cmap='RdBu_r', vmin=-abs_max, vmax=abs_max)
        cbar = plt.colorbar(im, ax=ax, label='pp diff', pad=0.02, shrink=0.8)
        cbar.ax.tick_params(labelsize=7)
        for i in range(10):
            for j in range(10):
                val = diff[i, j]
                if abs(val) > 1:
                    colour = 'white' if abs(val) > abs_max * 0.6 else 'black'
                    ax.text(j, i, f'{val:+.0f}', ha='center', va='center', color=colour, fontsize=6, fontweight='bold')
        for i in range(10):
            ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor='limegreen', linewidth=1.8))
        ax.set_xticks(range(10)); ax.set_xticklabels(range(1, 11), fontsize=7)
        ax.set_yticks(range(10)); ax.set_yticklabels(range(1, 11), fontsize=7)
        ax.set_xlabel('Retrieved Decile', fontsize=9, fontweight='bold')
        ax.set_ylabel('Expected Decile', fontsize=9, fontweight='bold')
        ax.set_title(f'{group}', fontsize=9, fontweight='bold')
    for idx in range(len(groups), n_rows*n_cols):
        r, c = divmod(idx, n_cols)
        axes[r, c].axis('off')
    plt.suptitle(f'Expected vs. Retrieved Diff by Company (BM25 − Approx, mode={DECILE_MODE})', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout()
    out = RESULTS_DIR / f'expected_vs_retrieved_heatmap_by_company_bm25_minus_approx.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out.name}')


### 7b. Expected vs. Retrieved Heatmap by Company

Break down the heatmap by company/dataset to reveal source-specific bias patterns.

In [ ]:
from src.metrics.metrics import pick_group_col as _pick_group_col
from src.metrics.bias_utils import build_heatmap_pct as _build_heatmap_pct

from src.metrics.decile_utils import decile_col_for, boundaries_for



decile_col = decile_col_for(DECILE_MODE)
boundaries = boundaries_for(DECILE_MODE, boundaries_uw, boundaries_cw)

for strategy in STRATEGIES:
    df = results_by_strategy[strategy]
    group_col = _pick_group_col(df)
    if not group_col: 
        print(f'⚠️ No grouping column for {strategy}')
        continue
    groups = sorted(df[group_col].unique())
    n_groups = len(groups)
    n_cols = min(3, n_groups)
    n_rows = (n_groups + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5*n_cols, 4.5*n_rows))
    if n_rows * n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)
    for idx, group in enumerate(groups):
        r, c = divmod(idx, n_cols)
        ax = axes[r, c]
        subset = df[df[group_col] == group]
        hm_pct = _build_heatmap_pct(subset, decile_col, boundaries)
        n_wrong = subset['wrong_docs_popularities'].apply(lambda x: len(x) if x else 0).sum()
        im = ax.imshow(hm_pct, aspect='auto', cmap='Reds', vmin=0, vmax=60)
        cbar = plt.colorbar(im, ax=ax, label='%', pad=0.02, shrink=0.8)
        cbar.ax.tick_params(labelsize=7)
        for i in range(10):
            for j in range(10):
                if hm_pct[i, j] > 1:
                    colour = 'white' if hm_pct[i, j] > 30 else 'black'
                    ax.text(j, i, f'{hm_pct[i, j]:.0f}', ha='center', va='center', color=colour, fontsize=6, fontweight='bold')
        for i in range(10):
            ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor='limegreen', linewidth=1.8))
        ax.set_xticks(range(10)); ax.set_xticklabels(range(1, 11), fontsize=7)
        ax.set_yticks(range(10)); ax.set_yticklabels(range(1, 11), fontsize=7)
        ax.set_xlabel('Retrieved Decile', fontsize=9, fontweight='bold')
        ax.set_ylabel('Expected Decile', fontsize=9, fontweight='bold')
        ax.set_title(f'{group}\n({n_wrong} wrong)', fontsize=9, fontweight='bold')
    for idx in range(len(groups), n_rows*n_cols):
        r, c = divmod(idx, n_cols)
        axes[r, c].axis('off')
    plt.suptitle(f'Expected vs. Retrieved by Company ({strategy.upper()}, mode={DECILE_MODE})', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout()
    out = RESULTS_DIR / f'expected_vs_retrieved_heatmap_by_company_{strategy}.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out.name}')


### 8. Popularity Preference Curve

For each expected-popularity decile: when the system retrieves a **wrong** document, how often is that document **more popular** than the correct answer?

$$P(\text{wrong} > \text{expected}) = \frac{\#\{\text{wrong docs with pop} > \text{target pop}\}}{\#\{\text{all wrong docs}\}}$$

Values above 0.5 indicate a systematic preference for popular content.

In [ ]:
from src.metrics.metrics import pick_group_col as _pick_group_col

# ============================================================================
# Popularity Preference Curve  - decile x-axis, per company/dataset panel
# ============================================================================

from src.metrics.decile_utils import decile_col_for

decile_col = decile_col_for(DECILE_MODE)

# Find grouping column using the first strategy that has data.
group_col = None
for strategy in STRATEGIES:
    if strategy in results_by_strategy and len(results_by_strategy[strategy]) > 0:
        group_col = _pick_group_col(results_by_strategy[strategy])
        if group_col is not None:
            break

if group_col is None:
    raise ValueError("No company/dataset grouping column found in results DataFrames.")

groups = set()
for strategy in STRATEGIES:
    if strategy in results_by_strategy and group_col in results_by_strategy[strategy].columns:
        vals = results_by_strategy[strategy][group_col].dropna().astype(str).tolist()
        groups.update(vals)
groups = sorted(groups)

if len(groups) == 0:
    raise ValueError(f"Grouping column {group_col!r} exists but contains no values.")

n_groups = len(groups)
n_cols = 2 if n_groups > 1 else 1
n_rows = int(np.ceil(n_groups / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4.8 * n_rows), squeeze=False)

x = range(1, 11)  # or list(range(1, 11))
top = np.linspace(1.0, 0.1, 10)
bottom = np.linspace(0.9, 0.0, 10)

for idx, grp in enumerate(groups):
    r, c = divmod(idx, n_cols)
    ax = axes[r][c]

    ax.fill_between(
        range(1, 11),
        top,
        bottom,
        color="black",
        alpha=0.15,
        label="Random Area",
        zorder=2,
    )

    for strategy in STRATEGIES:
        if strategy not in results_by_strategy:
            continue
        df = results_by_strategy[strategy]
        if group_col not in df.columns:
            continue

        sdf = df[df[group_col].astype(str) == grp]
        color = strategy_colors.get(strategy, "#333333")

        bias_per_decile = []
        ci95_per_decile = []
        n_per_decile = []

        for d in range(10):
            subset = sdf[sdf[decile_col] == d]
            n_higher = 0
            n_total = 0

            for wp_list, pop in zip(subset["wrong_docs_popularities"], subset[COL_POPULARITY]):
                if wp_list is None or len(wp_list) == 0 or pop is None:
                    continue
                for wp in wp_list:
                    n_total += 1
                    if wp > pop:
                        n_higher += 1

            if n_total > 0:
                p = n_higher / n_total
                ci95 = 1.96 * np.sqrt(p * (1 - p) / n_total)
            else:
                p = float("nan")
                ci95 = float("nan")

            bias_per_decile.append(p)
            ci95_per_decile.append(ci95)
            n_per_decile.append(n_total)

        valid = [i for i, v in enumerate(bias_per_decile) if not np.isnan(v)]
        if not valid:
            continue

        xs = [i + 1 for i in valid]
        ys = [bias_per_decile[i] for i in valid]
        err = [ci95_per_decile[i] for i in valid]
        ns = [n_per_decile[i] for i in valid]

        ax.errorbar(
            xs,
            ys,
            yerr=err,
            marker="o",
            linewidth=2.0,
            markersize=6.5,
            linestyle="-",
            color=color,
            capsize=3,
            markeredgewidth=1.0,
            markeredgecolor="white",
            label=strategy,
            zorder=6,
        )

        # Show sample count per point so uncertainty can be interpreted with n.
        for x, y, n in zip(xs, ys, ns):
            ax.annotate(
                f"n={n}",
                (x, y),
                textcoords="offset points",
                xytext=(0, 7),
                ha="center",
                fontsize=7.5,
                color=color,
                alpha=0.9,
            )

    ax.axhline(0.5, color="gray", linestyle=":", linewidth=1.2, alpha=0.55)
    ax.set_xlabel(f"Expected Popularity Decile ({DECILE_MODE})", fontsize=10.5, fontweight="bold")
    ax.set_ylabel("P(Wrong Doc More Popular)", fontsize=10.5, fontweight="bold")
    ax.set_title(f"{group_col}: {grp}", fontsize=12, fontweight="bold")
    ax.set_xticks(range(1, 11))
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.28)
    ax.legend(fontsize=8.5, framealpha=0.9, loc="best")

# Hide empty axes if grid has extra slots.
for j in range(n_groups, n_rows * n_cols):
    r, c = divmod(j, n_cols)
    axes[r][c].axis("off")

fig.suptitle(
    "Popularity Preference of Wrong Retrievals by Company/Dataset",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.savefig(
    os.path.join(str(RESULTS_DIR), "popularity_preference_decile_by_company.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("Saved -> popularity_preference_decile_by_company.png")


### 8c. Popularity Preference by Query Ambiguity (Entropy)

Splits queries into **high-entropy** (ambiguous — flat score distribution across top-K docs) and **low-entropy** (unambiguous — one doc clearly dominates) using the per-strategy median entropy as the threshold.

**Top row:** expected vs. retrieved popularity **heatmap** per entropy group (same layout as §7).  
**Bottom row:** **popularity preference curve** P(wrong doc more popular than target) per expected decile.  

A gap between the two groups indicates that query ambiguity modulates popularity bias.

In [ ]:
from src.metrics.metrics import pick_group_col as _pick_group_col
from src.metrics.bias_utils import build_heatmap_pct as _build_heatmap_pct, pref_curve as _pref_curve_entropy, render_heatmap as _render_heatmap

# ============================================================================
# 8c. Popularity Preference by Query Ambiguity — 5 entropy quintile bins
# ============================================================================

from src.metrics.decile_utils import decile_col_for, boundaries_for

decile_col = decile_col_for(DECILE_MODE)
boundaries = boundaries_for(DECILE_MODE, boundaries_uw, boundaries_cw)

ENTROPY_BINS = ["Q1_least", "Q2", "Q3", "Q4", "Q5_most"]
ENTROPY_LABELS = {
    "Q1_least": "Q1 least ambiguous",
    "Q2":       "Q2",
    "Q3":       "Q3",
    "Q4":       "Q4",
    "Q5_most":  "Q5 most ambiguous",
}
_cmap5 = plt.cm.RdYlBu_r
ENTROPY_COLORS = {b: _cmap5(i / 4) for i, b in enumerate(ENTROPY_BINS)}


_build_heatmap_pct_entropy = _build_heatmap_pct




# _render_heatmap imported from src.metrics.bias_utils

    ax.set_title(title, fontsize=8, fontweight="bold", color=color)


def _render_pref_overlay(ax, subsets_by_bin, decile_col, title):
    top = np.linspace(1.0, 0.1, 10)
    bot = np.linspace(0.9, 0.0, 10)
    ax.fill_between(range(1, 11), top, bot, color="black", alpha=0.10,
                    label="Random area", zorder=1)
    for bin_key, sub in subsets_by_bin.items():
        if sub is None or len(sub) == 0:
            continue
        bias, ci = _pref_curve_entropy(sub, decile_col)
        valid = [i for i, v in enumerate(bias) if not np.isnan(v)]
        if not valid:
            continue
        xs  = [i + 1 for i in valid]
        ys  = [bias[i] for i in valid]
        err = [ci[i] for i in valid]
        ax.errorbar(xs, ys, yerr=err, marker="o", linewidth=1.8, markersize=5,
                    linestyle="-", color=ENTROPY_COLORS[bin_key], capsize=3,
                    markeredgewidth=0.8, markeredgecolor="white",
                    label=ENTROPY_LABELS[bin_key], zorder=6)
    ax.axhline(0.5, color="gray", linestyle=":", linewidth=1.2, alpha=0.55)
    ax.set_xlabel(f"Expected Decile ({DECILE_MODE})", fontsize=8, fontweight="bold")
    ax.set_ylabel("P(Wrong > Target)", fontsize=8, fontweight="bold")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xticks(range(1, 11))
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.28)
    ax.legend(fontsize=7, framealpha=0.9, loc="best")


def _pick_group_col(df):
    for candidate in ["company", "provider", "dataset_company", "dataset_source",
                      "dataset_name", "dataset", "qa_dataset", "source_dataset"]:
        if candidate in df.columns:
            return candidate
    return ""


# ============================================================
# FIGURE A1 (all data): 5 heatmaps side by side, one per quintile
# FIGURE A2 (all data): preference curve overlay, all 5 bins
# ============================================================
for strategy in STRATEGIES:
    if strategy not in results_by_strategy:
        continue
    df = results_by_strategy[strategy]
    if "entropy_group" not in df.columns:
        print(f"\u26a0\ufe0f entropy_group missing for {strategy} \u2014 re-run compute metrics cell")
        continue

    subsets = {b: df[df["entropy_group"] == b] for b in ENTROPY_BINS}

    # A1: heatmaps
    fig, axes = plt.subplots(1, 5, figsize=(22, 5))
    for col_idx, bin_key in enumerate(ENTROPY_BINS):
        sub = subsets[bin_key]
        hm_pct = _build_heatmap_pct_entropy(sub, decile_col, boundaries)
        _render_heatmap(axes[col_idx], hm_pct,
                        f"{ENTROPY_LABELS[bin_key]}\n(n={len(sub):,})",
                        ENTROPY_COLORS[bin_key])
    fig.suptitle(
        f"Expected vs Retrieved Heatmaps by Entropy Quintile ({strategy.upper()}, mode={DECILE_MODE})",
        fontsize=13, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    out = RESULTS_DIR / f"heatmap_by_entropy_{strategy}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {out.name}")

    # A2: preference curves
    fig, ax = plt.subplots(figsize=(9, 5))
    _render_pref_overlay(ax, subsets, decile_col,
                         f"Popularity Preference by Entropy Quintile\n"
                         f"({strategy.upper()}, mode={DECILE_MODE})")
    plt.tight_layout()
    out = RESULTS_DIR / f"pref_curve_by_entropy_{strategy}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {out.name}")


# ============================================================
# FIGURE B1 (per company): heatmaps — rows=companies, cols=quintiles
# FIGURE B2 (per company): pref curves — one panel per company
# ============================================================
for strategy in STRATEGIES:
    if strategy not in results_by_strategy:
        continue
    df = results_by_strategy[strategy]
    if "entropy_group" not in df.columns:
        print(f"\u26a0\ufe0f entropy_group missing for {strategy}")
        continue

    group_col = _pick_group_col(df)
    if not group_col:
        print(f"\u26a0\ufe0f No company/dataset column found for {strategy}")
        continue

    companies = sorted(df[group_col].dropna().astype(str).unique())
    n_companies = len(companies)

    # B1: heatmaps — rows=companies, cols=quintiles
    fig, axes = plt.subplots(n_companies, 5,
                             figsize=(22, 5 * n_companies),
                             squeeze=False)
    for row_idx, company in enumerate(companies):
        sub_all = df[df[group_col].astype(str) == company]
        subsets = {b: sub_all[sub_all["entropy_group"] == b] for b in ENTROPY_BINS}
        for col_idx, bin_key in enumerate(ENTROPY_BINS):
            sub = subsets[bin_key]
            hm_pct = _build_heatmap_pct_entropy(sub, decile_col, boundaries)
            _render_heatmap(
                axes[row_idx, col_idx], hm_pct,
                f"{company} \u2014 {ENTROPY_LABELS[bin_key]}\n(n={len(sub):,})",
                ENTROPY_COLORS[bin_key],
            )
    fig.suptitle(
        f"Heatmaps by Company & Entropy Quintile ({strategy.upper()}, mode={DECILE_MODE})",
        fontsize=14, fontweight="bold", y=1.005,
    )
    plt.tight_layout()
    out = RESULTS_DIR / f"heatmap_by_entropy_per_company_{strategy}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {out.name}")

    # B2: preference curves — one panel per company
    n_cols_pref = min(3, n_companies)
    n_rows_pref = (n_companies + n_cols_pref - 1) // n_cols_pref
    fig, axes = plt.subplots(n_rows_pref, n_cols_pref,
                             figsize=(9 * n_cols_pref, 5 * n_rows_pref),
                             squeeze=False)
    for idx, company in enumerate(companies):
        r, c = divmod(idx, n_cols_pref)
        sub_all = df[df[group_col].astype(str) == company]
        subsets = {b: sub_all[sub_all["entropy_group"] == b] for b in ENTROPY_BINS}
        _render_pref_overlay(
            axes[r, c], subsets, decile_col,
            f"{company}",
        )
    for idx in range(n_companies, n_rows_pref * n_cols_pref):
        r, c = divmod(idx, n_cols_pref)
        axes[r, c].axis("off")
    fig.suptitle(
        f"Popularity Preference Curves by Company & Entropy Quintile ({strategy.upper()}, mode={DECILE_MODE})",
        fontsize=14, fontweight="bold", y=1.005,
    )
    plt.tight_layout()
    out = RESULTS_DIR / f"pref_curve_by_entropy_per_company_{strategy}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {out.name}")


In [ ]:
# ============================================================================
# 8d. Entropy Quintile Distribution Across Popularity Deciles — Stacked Bar Chart
# ============================================================================
# For each strategy: one stacked bar chart per dataset.
# X-axis = popularity decile (1–10).
# Y-axis = fraction of questions in that decile.
# Each bar is stacked by entropy quintile group (Q1_least … Q5_most).
# Shows whether ambiguous vs. unambiguous queries are evenly distributed
# across popularity deciles, or concentrated in certain deciles.
# ============================================================================

from src.metrics.decile_utils import decile_col_for

decile_col = decile_col_for(DECILE_MODE)


def _plot_entropy_dist_stacked(
    df: pd.DataFrame,
    *,
    decile_col: str,
    title: str,
    ax,
) -> None:
    """Render one stacked bar chart on *ax*.

    Each bar covers one popularity decile (1–10). Segments show the
    fraction of questions in that decile belonging to each entropy
    quintile (Q1_least … Q5_most).
    """
    deciles = list(range(10))
    # Build a (10 x 5) count matrix
    counts = pd.DataFrame(0, index=deciles, columns=ENTROPY_BINS)
    for d in deciles:
        sub = df[df[decile_col] == d]
        for b in ENTROPY_BINS:
            counts.at[d, b] = (sub["entropy_group"] == b).sum()

    # Normalise rows to fractions
    row_totals = counts.sum(axis=1).replace(0, np.nan)
    fracs = counts.div(row_totals, axis=0).fillna(0.0)

    x = np.arange(10)
    bottom = np.zeros(10)
    for b in ENTROPY_BINS:
        vals = fracs[b].values
        ax.bar(
            x,
            vals,
            bottom=bottom,
            color=ENTROPY_COLORS[b],
            label=ENTROPY_LABELS[b],
            width=0.75,
            edgecolor="white",
            linewidth=0.4,
        )
        bottom += vals

    ax.axhline(0.2, color="black", linestyle="--", linewidth=0.9, alpha=0.45,
               label="Uniform (20 %)")
    ax.set_xticks(x)
    ax.set_xticklabels([str(d + 1) for d in deciles], fontsize=8)
    ax.set_xlabel(f"Popularity Decile ({DECILE_MODE})", fontsize=8, fontweight="bold")
    ax.set_ylabel("Fraction of Questions", fontsize=8, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(fontsize=6, framealpha=0.85, loc="upper right", ncol=2)

    # Annotate each bar with total question count
    for d in deciles:
        n = int(row_totals.iloc[d]) if not np.isnan(row_totals.iloc[d]) else 0
        ax.text(d, 1.01, str(n), ha="center", va="bottom", fontsize=6, color="#333")


for strategy in STRATEGIES:
    if strategy not in results_by_strategy:
        continue
    df = results_by_strategy[strategy]
    if "entropy_group" not in df.columns:
        print(f"\u26a0\ufe0f entropy_group missing for {strategy} — re-run compute metrics cell")
        continue

    group_col = _pick_group_col(df)

    # ── All-data chart (no dataset split) ────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 4.5))
    _plot_entropy_dist_stacked(
        df,
        decile_col=decile_col,
        title=(
            f"Entropy Quintile Distribution per Popularity Decile\n"
            f"({strategy.upper()}, mode={DECILE_MODE}, n={len(df):,})"
        ),
        ax=ax,
    )
    plt.tight_layout()
    out = RESULTS_DIR / f"entropy_dist_per_decile_{strategy}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {out.name}")

    # ── Per-dataset charts ────────────────────────────────────────────────
    if not group_col:
        print(f"\u26a0\ufe0f No dataset/company column found for {strategy} — skipping per-dataset plots")
        continue

    datasets = sorted(df[group_col].dropna().astype(str).unique())
    n_ds = len(datasets)

    # Lay out in a grid: up to 2 columns
    n_cols = min(2, n_ds)
    n_rows = (n_ds + n_cols - 1) // n_cols
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(10 * n_cols, 4.5 * n_rows),
        squeeze=False,
    )

    for idx, dataset in enumerate(datasets):
        r, c = divmod(idx, n_cols)
        sub = df[df[group_col].astype(str) == dataset]
        _plot_entropy_dist_stacked(
            sub,
            decile_col=decile_col,
            title=f"{dataset} (n={len(sub):,})",
            ax=axes[r, c],
        )

    # Hide unused axes
    for idx in range(n_ds, n_rows * n_cols):
        r, c = divmod(idx, n_cols)
        axes[r, c].axis("off")

    fig.suptitle(
        f"Entropy Quintile Distribution per Popularity Decile — By Dataset\n"
        f"({strategy.upper()}, mode={DECILE_MODE})",
        fontsize=13, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    out = RESULTS_DIR / f"entropy_dist_per_decile_by_dataset_{strategy}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {out.name}")
